## Importing and Initial Variables

In [ ]:
# from todoist_api_python.api import TodoistAPI
import pandas as pd
from utils import credentials as cd, config as config
from dateutil import parser as dateparser
import requests
from datetime import datetime
from pathlib import Path

COLUMN_TEMPLATE = config.TODOIST_COLUMNS
FILENAME = "my-energysystem-tasks_raw.csv"
mostrecentdate = None
df = None
data_path = Path(Path.cwd(),r"data/",FILENAME)
baseurl = "https://api.todoist.com/api/v1/"
get_tasks = "tasks/completed/by_completion_date/"
get_projects = "projects/"

headers = {"Authorization": f"Bearer {cd.api_key}"}

request_body = {
    "since" : None,
    "until" : None,
    "limit": 200
}

In [ ]:
Path.exists(data_path)

## Functions

In [ ]:
def isoToString(date, format):
    input_date = dateparser.parse(date)
    return input_date.strftime(format=format)

def column_is_type(df):
    return df.transform(lambda x: x.apply(type)).drop_duplicates().iloc[0]

def split_column(data,split,key):
    test = data[[key,split]]
    test = test[test.iloc[:, 1].notna()]
    # print(column_is_type(test.iloc[:,1]))
    if column_is_type(test.iloc[:,1]) is dict:
        unnest = test[split].apply(pd.Series)
        source_names = unnest.columns.tolist()
        updated_names = [split.capitalize() + part.capitalize() for part in source_names]

        rename_zip = zip(source_names, updated_names)
        rename_dict = dict(rename_zip)

        renamed = unnest.rename(rename_dict, axis=1)
        output = test.join(renamed)
        return output

# def split_column(data,split, key):
#     test = data[[key[1],split]]
#     test = test[test.iloc[:, 1].notna()]
#     # print(test.info())
#     # print(column_is_type(test.iloc[:,1]))
#     if column_is_type(test.iloc[:,1]) is dict:
#         unnest = test[split].apply(pd.Series)
#         # drop_last_col = len(unnest.columns)
#         # unnest = unnest.iloc[:,0:drop_last_col-1]
#         # print(unnest.info())
#         source_names = unnest.columns.tolist()
#         # print("sourcename: {}".format(source_names))
#         updated_names = [split.capitalize() + str(part).capitalize() for part in source_names]

#         rename_zip = zip(source_names, updated_names)
#         rename_dict = dict(rename_zip)

#         renamed = unnest.rename(rename_dict, axis=1)
#         output = test.join(renamed)
#         return output

def todistRequest(request, param, header):
    items = []
    cursor = None
    url = baseurl + request
    while True:
        params = param
        if cursor:
            params["cursor"] = cursor

        response = requests.get(url, headers=header, params=params)
        response.raise_for_status()
        data = response.json()

        items.extend(data["results"])

        cursor = data.get("next_cursor")
        if not cursor:
            break
    return items

## Check for existing archive of tasks

In [ ]:
# Path.read_bytes(data_path)
if Path.exists(data_path) is False:
    print("file not found.\npull from earliest month start.")
else:
    print("file found. Opening to collect last task date")
    df = pd.read_csv(data_path).infer_objects()
    mostrecentdate = df['completed_at'].max()
    

### Grab the last completed date as a starting point to pull more tasks

In [ ]:
end_date = datetime.now()

if mostrecentdate is None:
    print("No most recent date found")
    start_date = datetime.now()
    start_date = datetime.replace(start_date,day=1)
else:    
    mostrecentdate = dateparser.parse(mostrecentdate)
    start_date = mostrecentdate
    
iso_start_date = start_date.isoformat()
iso_end_date = end_date.isoformat()
    # datetime.strptime("5/1/2026","%m/%d/%Y")
# end_date = datetime.strptime("5/18/2026","%m/%d/%Y")



## Collect Project and Tasks

In [ ]:
request_body['since'] = start_date
request_body['until'] = end_date

In [ ]:
project_request = request_body.get("limit")

projects = todistRequest(get_projects, request_body, headers)
proj_df = pd.DataFrame(projects)
proj_df = proj_df[['id','name']]

In [ ]:
items = todistRequest(get_tasks,request_body,headers)

In [ ]:
# response = requests.get(url, headers=headers,params = request_body)

In [ ]:
# data = response.json()

# items = data.get("items",[])

# len(items)

## Build Task Table

In [ ]:
# def dataframeIndexDict(df):
#     columns = df.columns
#     idx = 0
#     column_dict = dict()

#     for c in columns.to_list():
#         column_dict[idx] = c
#         idx += 1
#     return column_dict

In [ ]:
task_table = pd.DataFrame(items)

if df is not None:
    task_table = pd.concat([task_table,df])

task_table = task_table.reset_index(drop=True)
# task_table


In [ ]:
id_column = (task_table.columns.get_loc("id"), "id")

## Split due date iterable into unique columns

In [ ]:
task_table['due'].apply(type).value_counts()

In [ ]:
date_columns = split_column(data=task_table,split='due',key=id_column[1])

# date_columns
if date_columns is not None:
    combined_df = pd.merge(left=task_table, right=date_columns, on=id_column[1],how='left')
# combined_df

In [ ]:
# combined_df['added_at_test'] = [isoToString(row,"%Y/%m/%d") for row in combined_df['added_at']]

In [ ]:
dates_to_convert = ['added_at', 'completed_at','updated_at']

combined_df[dates_to_convert] = combined_df[dates_to_convert].apply(lambda row: [isoToString(rowItem,"%Y/%m/%d") for rowItem in row])

In [ ]:
combined_df

## Drop Raw data & Unneeded columns

In [ ]:
final_df = combined_df.drop(columns=COLUMN_TEMPLATE['drop'], errors="ignore")

final_df = final_df.dropna(axis=1,how="all")
final_df.info()

In [ ]:
final_df